intro:
(as per discussion post)

glossary of fields, which get encoded and how and why

plan:
-preprocess
-encode
-supporting charts to understand data, distributions etc
    - convert percentages
    - distributions graphs
    - exit rates
-logistic regression (explain why this model and others..strenghts/weaknesses)
-descision tree
-random forest
-sampling or way to make data more accurate, bootstrapping?




analysis:
-with and without pagevalues
-f1 score, confusion matrix
-aic and bic?  no roc/auc

conclusion:
- discuss conclusions 
- model performance, are the more complex better, did we tune them, why

### Final Project - Online Shoppers Purchasing Intention Dataset

Dataset main page: [here](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset)

Dataset zip file: [here](https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip)


The dataset is from the *UCI Machine Learning Repository*.  The data models user behavior on a popular e-commerce site to predict the intent to make a purchase.

Note: The feature metrics relate to user behavior, such as time on a page, page type visited, time of year (special events), and bounce and exit rates. 

**Objective:**
- The question we are trying to solve is: **will a session result in a purchase.**
- This information is valuable because it can reveal which parts of the user journey on the e-commerce site add friction to the user's experience, thus causing undesirable behavior (high bounce/exit rates or no purchase).
- The strength of this data is that it models user behavior

**Dataset Details:**

- Contains feature vectors from 12,300 user sessions
- Each session belongs to a different user over a 1-year period, which helps avoid behavioral bias toward events, periods, or a specific user profile.
- The target variable is 'Revenue' and is labeled as a boolean (true/false), making the dataset appropriate for a supervised classification model.

- The model is a classification model because the target variable 'Revenue' is binary (purchased or not purchased).
- There are a number of categorical features that will be converted using **One Hot Encoding**:
    - Month (month of year)
    - OperatingSystems (type of OS the user is on)
    - Browser (type of browser the user is on)
    - Region (geographic region)
    - TrafficType (how did the user arrive here: search, referral, other)
    - VisitorType (new visitor, returning)
    - Weekend (boolean)
- The other features are continuous and will need to be scaled so that models (specifically for Logistic Regression) do not give too much weight to larger values 
    - page counts (Administrative, Informational, ProductRelated)
    - page durations (Administrative, Informational, ProductRelated)
    - bounce rate (% of 1 page view sessions)
    - exit rate (% of a page being the last in a session)
    - page value (page rank of a page if the session converted - consider dropping due to data leakage)
    - special day score (rank of how close the session is to a holiday)



**Model Selection:**

_Logistic Regression_ will provide a logical choice for a baseline model.  It works well for this binary classification model.  It can show which features are associated with a higher/lower probability of a purchase.

_Decision Trees_ will be interesting to compare with Logistic Regression, a decision tree can show how feature combinations result in a purchase or non-purchase.  Seeing this visually in a tree diagram will make it easy to interpret.

Since we're using Decision Trees, we should go a step further and run the data through a _Random Forest_.  This will be beneficial by reducing overfitting.  Some relationships may be complex resulting in the model learning 'too well' and thus overfitting.

**Data Exploration**
Provide a series of graphs to help visualize how the data is generally structured.  These graphs will not fully explain the model, but they indicate which features influence the outcome the most.  And they intuitively describe user behavior in the real world


**Feature Selection**

We'll use the VIF — Variance Inflation Factor to check for multicollinearity.  For simplicity sake, once features are identified as redundant, we'll drop for all models.  This keeps the comparison between models simpler and more consistent


**Evaluation**

Model performance will be evaluated using a confusion matrix, accuracy, precision, recall, and F1-score. Since purchases may be less common than non-purchases, I will not rely only on accuracy. I will likely prioritize F1-score because it balances precision and recall.

##### Setup imports 

In [1]:
# setup standard imports for data analysis and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
import statsmodels.api as sm

# modeling and evaluation
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

##### Step 1 - Data Cleaning

We can see our data is in good shape, no missing values pre column and a typical split on revenue across the data set at about 15% purchase, 85% do not purchase.

In [16]:
# read data in a global variable so it can be used in other cells


data_path = "online-shoppers-intention.csv"
df = pd.read_csv(data_path)
target_column = "Revenue"
# make a working copy and convert boolean style values into numeric form to match the other columns and make it easier to work with
working_df = df.copy()
working_df["Weekend"] = working_df["Weekend"].astype(str).str.upper().map({"TRUE": 1, "FALSE": 0})
working_df["Revenue"] = working_df["Revenue"].astype(str).str.upper().map({"TRUE": 1, "FALSE": 0})


duplicate_rows = working_df.duplicated().sum()
working_df = working_df.drop_duplicates().reset_index(drop=True)

print("missing values by column:")
print(working_df.isna().sum())

print("duplicate rows removed:", duplicate_rows)
print("\n")
print(f"total rows: {working_df.shape[0]}")
print("\n")
print(f"total columns: {working_df.shape[1]}")
print("\n")
print("class balance:")
print(working_df[target_column].value_counts(normalize=True).rename("proportion"))


missing values by column:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64
duplicate rows removed: 125


total rows: 12205


total columns: 18


class balance:
Revenue
0    0.843671
1    0.156329
Name: proportion, dtype: float64


##### Step 2 - One Hot Encode Data

 One hot encode the categorical values: (_Month, OperatingSystems, Browser, Region, TrafficType, and VisitorType_)

 We can see our colunn count goes from 18 to 75
 We have 20 unique columns for Traffic Type
 A snippet of the first 5 rows of the data is shown with all columns including the one hot ecoded ones.

In [26]:
# one hot encode categorical columns
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
]

columns_to_encode = [
    column for column in categorical_features if column in working_df.columns
]

if columns_to_encode:
    working_df = pd.get_dummies(
        working_df,
        columns=columns_to_encode,
        dtype=int,
    )

encoded_df = working_df.copy()

print(f"encoded shape: {encoded_df.shape}")
with pd.option_context("display.max_columns", None):
    display(encoded_df.head())

encoded shape: (12205, 75)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Weekend,Revenue,Month_Aug,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,OperatingSystems_1,OperatingSystems_2,OperatingSystems_3,OperatingSystems_4,OperatingSystems_5,OperatingSystems_6,OperatingSystems_7,OperatingSystems_8,Browser_1,Browser_2,Browser_3,Browser_4,Browser_5,Browser_6,Browser_7,Browser_8,Browser_9,Browser_10,Browser_11,Browser_12,Browser_13,Region_1,Region_2,Region_3,Region_4,Region_5,Region_6,Region_7,Region_8,Region_9,TrafficType_1,TrafficType_2,TrafficType_3,TrafficType_4,TrafficType_5,TrafficType_6,TrafficType_7,TrafficType_8,TrafficType_9,TrafficType_10,TrafficType_11,TrafficType_12,TrafficType_13,TrafficType_14,TrafficType_15,TrafficType_16,TrafficType_17,TrafficType_18,TrafficType_19,TrafficType_20,VisitorType_New_Visitor,VisitorType_Other,VisitorType_Returning_Visitor
0,-0.702302,-0.460019,-0.398824,-0.246257,-0.696218,-0.628793,3.969402,3.434394,-0.318962,-0.31024,-0.553088,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,-0.702302,-0.460019,-0.398824,-0.246257,-0.673793,-0.595451,-0.450137,1.268054,-0.318962,-0.31024,-0.553088,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,-0.702302,-0.460019,-0.398824,-0.246257,-0.696218,-0.628793,3.969402,3.434394,-0.318962,-0.31024,-0.553088,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,-0.702302,-0.460019,-0.398824,-0.246257,-0.673793,-0.627404,0.654748,2.134590,-0.318962,-0.31024,-0.553088,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,-0.702302,-0.460019,-0.398824,-0.246257,-0.494387,-0.301889,-0.008183,0.184884,-0.318962,-0.31024,1.808031,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


##### Step 3 - Normalize numeric features

In [27]:
numeric_features = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Weekend",
]

target = "Revenue"
y = encoded_df[target].copy()
X_tree = encoded_df.drop(columns=[target]).copy()
X_logistic = X_tree.copy()

scaler = StandardScaler()
X_logistic[numeric_features] = scaler.fit_transform(X_logistic[numeric_features])

print(f"X_tree shape: {X_tree.shape}")
print(f"X_logistic shape: {X_logistic.shape}")
print(f"y shape: {y.shape}")
X_logistic[numeric_features].head()

X_tree shape: (12205, 74)
X_logistic shape: (12205, 74)
y shape: (12205,)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Weekend
0,-0.702302,-0.460019,-0.398824,-0.246257,-0.696218,-0.628793,3.969402,3.434394,-0.318962,-0.31024,-0.553088
1,-0.702302,-0.460019,-0.398824,-0.246257,-0.673793,-0.595451,-0.450137,1.268054,-0.318962,-0.31024,-0.553088
2,-0.702302,-0.460019,-0.398824,-0.246257,-0.696218,-0.628793,3.969402,3.434394,-0.318962,-0.31024,-0.553088
3,-0.702302,-0.460019,-0.398824,-0.246257,-0.673793,-0.627404,0.654748,2.134590,-0.318962,-0.31024,-0.553088
4,-0.702302,-0.460019,-0.398824,-0.246257,-0.494387,-0.301889,-0.008183,0.184884,-0.318962,-0.31024,1.808031
